   This Python script is the visual companion to the C telemetry logger you
   ... WHile your C code runs the actual high-performance benchmark and writes
   the `write_trace_row` CSV data, this Python script reads that exact CSV file
   and replays the rollout inside a 3D viewer so you can watch what actually
   happened. Think of it as a "flight data recorder" playback tool for your
   robot arm.

   The `load_trace` and `apply_ctrl` functions handle the data plumbing. The
   script reads your CSV and specifically hunts for the headers you geneated 
   that starts with `ctrl_` (your raw motor commands). For every row in the file
   , `apply_ctrl` takes those saved numbers and forcefully injects them directly
   into the live MuJoCo `data.ctrl` array. Instead of an active controller
   calculating math to decide what to do, Python is just "puppeteering" the
   robot by feeding it historical electrical signals frame-by-frame.

   The `replay_view` function is where the 3D rendering happens. It boots up the
   official MuJoCo interactive viewer. Inside the loop, it applies the recorded
   controls, steps the physics, and then calls `viewer.sync()` to update the 
   pixels on your screen. THe most importatn logic here is the `delay` 
   calculation. Becuase your CPU can simulate physics thousands of times faster 
   than real-time, the script calculates the exact timestamp difference between
   the CSV rows and uses `time.sleep()` to artificially puase the loop. This
   ensures you watch the robot move at a normal, lifelike speed rather than 
   having it instantly snap to the end of the trial in a single blur.

   FInally, the `main` function acts as your CLI, giving you some handly flags.
   You can use `--speed 0.5` to watch a tricky socket insertion in slow-motion,
   or use `--headless` to trigger the `replay_headless` function. Headless mode
   strips away the 3D graphics entirely and just crunches the physics math as 
   fast as your orocessor allows. This is perfect if you just want the computer 
   to silently verify that a trace file isn't corrupted without aking you sit
   there and watch the whole animation playout.